# 📊 Exploratory Data Analysis & Production Bridge

> **Pedagogical Goal:** Demystify how data science code in a Jupyter Notebook transitions into clean, modular, production-ready backend services for a **FastAPI** application.

### The 4-Step Engineering Workflow:
1. **Explore & Understand:** Load `sales_data.csv` and inspect distributions.
2. **Prototype Logic:** Compute KPIs (Total Revenue, Orders, Average Order Value, Category breakdown).
3. **Refactor to Pure Functions:** Move logic into `app/services.py` for reusability and unit testing.
4. **Expose as REST Endpoints:** Deliver results via typed FastAPI routes in `app/main.py`.

## 1. Load and Inspect the Dataset

We begin by reading the generated CSV transaction dataset using **Pandas**.

In [1]:
from pathlib import Path

import pandas as pd

# Resolve path to sales_data.csv inside the app directory
data_path = Path("../app/sales_data.csv")

df = pd.read_csv(data_path)
print(f"Loaded {len(df)} transactions.")
df.head()

Loaded 500 transactions.


,Transaction_ID,Date,Category,Unit_Price,Quantity
0,TXN-1001,2026-09-14,Home,177.90,4
1,TXN-1002,2026-08-29,Home,329.64,4
2,TXN-1003,2026-08-29,Home,119.30,2
3,TXN-1004,2026-08-21,Toys,445.81,4
4,TXN-1005,2026-08-24,Electronics,385.02,5


### Inspect Data Types and Missing Values
Checking for null values and verifying column data types (`Transaction_ID`, `Date`, `Category`, `Unit_Price`, `Quantity`).

In [2]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Transaction_ID  500 non-null    str    
 1   Date            500 non-null    str    
 2   Category        500 non-null    str    
 3   Unit_Price      500 non-null    float64
 4   Quantity        500 non-null    int64  
dtypes: float64(1), int64(1), str(3)
memory usage: 19.7 KB


,Unit_Price,Quantity
count,500.000000,500.000000
mean,253.590480,3.182000
std,142.769533,1.358148
min,12.160000,1.000000
25%,122.522500,2.000000
50%,243.915000,3.000000
75%,380.510000,4.000000
max,499.900000,5.000000


## 2. Feature Engineering: Total Sales Calculation

Each transaction line item represents a quantity of items sold at a specific unit price.
We compute the line item total revenue using vectorized Pandas arithmetic:
$$\text{Total Sales} = \text{Quantity} \times \text{Unit Price}$$

In [3]:
df["Total_Sales"] = df["Quantity"] * df["Unit_Price"]
df[["Transaction_ID", "Category", "Unit_Price", "Quantity", "Total_Sales"]].head()

,Transaction_ID,Category,Unit_Price,Quantity,Total_Sales
0,TXN-1001,Home,177.90,4,711.60
1,TXN-1002,Home,329.64,4,1318.56
2,TXN-1003,Home,119.30,2,238.60
3,TXN-1004,Toys,445.81,4,1783.24
4,TXN-1005,Electronics,385.02,5,1925.10


## 3. Prototype Business KPIs (Key Performance Indicators)

Now we prototype the summary analytics required by executive dashboards:
- **Total Gross Revenue**
- **Total Order Count**
- **Average Order Value (AOV)**

In [4]:
total_revenue = float(df["Total_Sales"].sum())
total_orders = len(df)
average_order = total_revenue / total_orders if total_orders > 0 else 0.0

summary_kpis = {
    "total_revenue": round(total_revenue, 2),
    "total_orders": total_orders,
    "average_order_value": round(average_order, 2)
}

print("Computed Summary KPIs:")
for k, v in summary_kpis.items():
    print(f"  • {k}: {v}")

Computed Summary KPIs:
  • total_revenue: 394974.25
  • total_orders: 500
  • average_order_value: 789.95


## 4. Categorical Breakdown

We aggregate sales performance by category to identify our top revenue drivers.

In [5]:
category_revenue = df.groupby("Category")["Total_Sales"].sum().round(2).to_dict()
category_revenue

{'Clothing': 98689.93,
 'Electronics': 82675.89,
 'Home': 93051.37,
 'Toys': 120557.06}

In [6]:
# Frequency and revenue table per category
category_summary = df.groupby("Category").agg(
    Orders=("Transaction_ID", "count"),
    Total_Quantity=("Quantity", "sum"),
    Gross_Revenue=("Total_Sales", "sum")
).round(2).sort_values(by="Gross_Revenue", ascending=False)

category_summary

,Orders,Total_Quantity,Gross_Revenue
Category,,,
Toys,141,465,120557.06
Clothing,124,398,98689.93
Home,118,372,93051.37
Electronics,117,356,82675.89


## 5. 🌉 The Bridge: Refactoring from Notebook to Production

### Why do we refactor?
| Notebook Code (Data Science) | Production Service (`app/services.py`) |
| :--- | :--- |
| Relies on global state (`df`) | Accepts parameters explicitly (Pure functions) |
| Runs cell-by-cell interactively | Packaged into importable, reusable modules |
| Difficult to automate with tests | Easily unit-tested using `pytest` |
| Outputs directly to console/plots | Returns structured dictionaries / Pydantic models |

Here is how our notebook calculations map directly into `app/services.py`:

In [7]:
import sys

# Add the app directory to path so we can import our production code
app_dir = str(Path("../app").resolve())
if app_dir not in sys.path:
    sys.path.insert(0, app_dir)

# Import the production functions
from services import get_category_revenue, get_summary_metrics, load_data

# Verify that our production functions match notebook calculations
prod_df = load_data(str(data_path))
prod_summary = get_summary_metrics(prod_df)
prod_categories = get_category_revenue(prod_df)

print("Production Services Output:")
print("Summary Metrics:", prod_summary)
print("Category Revenue:", prod_categories)

# Assert that production calculations match exploratory findings
assert prod_summary == summary_kpis
assert prod_categories == category_revenue
print("\n✅ All production outputs match notebook exploration perfectly!")

Production Services Output:
Summary Metrics: {'total_revenue': 394974.25, 'total_orders': 500, 'average_order_value': 789.95}
Category Revenue: {'Clothing': 98689.93, 'Electronics': 82675.89, 'Home': 93051.37, 'Toys': 120557.06}

✅ All production outputs match notebook exploration perfectly!


## 6. How FastAPI Delivers This to Web Clients

In `app/main.py`, FastAPI exposes these functions via HTTP endpoints:

```python
from fastapi import FastAPI
from schemas import SummaryMetricsResponse
from services import get_summary_metrics

app = FastAPI()

@app.get("/metrics/summary", response_model=SummaryMetricsResponse)
def summary():
    return get_summary_metrics(df)
```

When a browser or client requests `GET /metrics/summary`:
1. FastAPI invokes `get_summary_metrics(df)` from `services.py`.
2. The returned dictionary is validated against `SummaryMetricsResponse` in `schemas.py`.
3. FastAPI serializes it into clean JSON HTTP response.